# Bloque 1: Fundamentos de Modelos y Mensajes

Para construir agentes autónomos con LangGraph, primero debemos entender la unidad más básica de LangChain: la comunicación directa con el Modelo de Lenguaje (LLM). 

En este notebook aprenderemos:
1. Cómo configurar el cliente del LLM.
2. Los 3 tipos de mensajes fundamentales (`System`, `Human`, `AI`).
3. La anatomía de una respuesta cuando el modelo decide usar una herramienta (Tool Calling).

In [ ]:
import os
from dotenv import load_dotenv

# Cargar variables de entorno (Asegúrate de tener tu archivo .env configurado con OPENAI_API_KEY)
load_dotenv(dotenv_path="../.env")

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️ Error: No se encontró OPENAI_API_KEY. Revisa tu archivo .env")
else:
    print("✅ Entorno configurado correctamente.")

## 1. Chat Models y la Lista de Mensajes

En LangChain, no enviamos una simple cadena de texto al modelo. Le enviamos una **lista de mensajes** estructurada. Los principales son:
* `SystemMessage`: Define el comportamiento, contexto o "personalidad" del sistema (el prompt del sistema).
* `HumanMessage`: La entrada o petición del usuario.
* `AIMessage`: La respuesta generada por el modelo.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Instanciamos el modelo. 
# temperature=0 es vital para los agentes: queremos respuestas lógicas y deterministas, no creativas.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Construimos el historial/contexto
mensajes = [
    SystemMessage(content="Eres un asistente de base de datos corporativo. Responde de manera muy breve."),
    HumanMessage(content="¿Qué es una clave primaria en SQL?")
]

# Invocamos al modelo pasándole la lista
respuesta_normal = llm.invoke(mensajes)

print("--- Respuesta del Modelo ---")
print(f"Tipo de objeto: {type(respuesta_normal)}")
print(f"Contenido: {respuesta_normal.content}")

## 2. Preparando al Modelo para usar Herramientas

Cuando construyamos nuestro agente, el LLM no siempre responderá con texto. A veces, decidirá que necesita ejecutar una acción externa. 

Para que el modelo sepa qué acciones puede tomar, le "presentamos" las herramientas usando `.bind_tools()`. Observa cómo cambia la estructura de la respuesta (`AIMessage`) cuando el modelo decide usar una de ellas: el `.content` se queda vacío y se llena el atributo `.tool_calls`.

In [ ]:
# Definimos una función de Python falsa solo como ejemplo
def buscar_cliente(nombre: str) -> str:
    """Busca la información de contacto de un cliente en la base de datos."""
    pass

# "Conectamos" la herramienta al modelo. Esto le inyecta las instrucciones
# de la función a la API del LLM.
llm_con_herramientas = llm.bind_tools([buscar_cliente])

# Hacemos una petición que claramente requiere usar la herramienta
mensajes_accion = [
    SystemMessage(content="Eres un asistente. Si te piden datos de clientes, usa la herramienta buscar_cliente."),
    HumanMessage(content="¿Me puedes dar el teléfono de Ana Lopez?")
]

respuesta_accion = llm_con_herramientas.invoke(mensajes_accion)

print("--- Anatomía de un Tool Call ---")
print(f"¿Tiene texto la respuesta? -> '{respuesta_accion.content}' (Vacío)")
print("\nEl modelo generó una petición estructurada en 'tool_calls':")
for tool_call in respuesta_accion.tool_calls:
    print(f"- Herramienta solicitada: {tool_call['name']}")
    print(f"- Argumentos extraídos por el LLM: {tool_call['args']}")